[![Roboflow Notebooks](https://media.roboflow.com/notebooks/template/bannertest2-2.png?ik-sdk-version=javascript-1.4.3&updatedAt=1672932710194)](https://github.com/roboflow/notebooks)

# Train Cricket Pitch Keypoint Detector

---

This notebook trains a YOLOv8 pose model that finds the creases and the stumps in
broadcast footage filmed from behind the bowler. The model gives the points needed to
compute the homography between the image and the pitch, which replaces the manual
calibration in `examples/cricket/pitch_map.py`.

It merges three public Roboflow Universe datasets that label the same real-world points
with different names. All are licensed CC BY 4.0, so credit the authors if you publish
a model trained on them.

| dataset | author | images used |
|:--------|:-------|:-----------:|
| [cricket-pitch-keypoints](https://universe.roboflow.com/nishith-n6waq/cricket-pitch-keypoints) | nishith-n6waq | 393 |
| [cricket-pitch-detection](https://universe.roboflow.com/maree/cricket-pitch-detection-wdtth) | maree | 53 |
| [cricket-pitch-keypoint](https://universe.roboflow.com/adrian-charan/cricket-pitch-keypoint-kbktj) | adrian-charan | 224 |

The merged dataset has 14 keypoints: the 4 corners (bowling crease x return crease), the
4 popping crease points (popping crease x return crease) and the base of the 3 stumps at
each end. See `examples/cricket/prepare_pitch_dataset.py` for the exact mapping.

## Configure API keys

- Open your [`Roboflow Settings`](https://app.roboflow.com/settings/api) page. Click `Copy`. This will place your private key in the clipboard.
- In Colab, go to the left pane and click on `Secrets` (🔑). Store Roboflow API Key under the name `ROBOFLOW_API_KEY`.

## Before you start

Let's make sure that we have access to GPU. We can use `nvidia-smi` command to do that. In case of any problems navigate to `Edit` -> `Notebook settings` -> `Hardware accelerator`, set it to `GPU`, and then click `Save`.

In [ ]:
!nvidia-smi

In [ ]:
import os
HOME = os.getcwd()
print(HOME)

## Install dependencies

In [ ]:
!pip install -q ultralytics roboflow supervision

# `sports` is used from source. Point these to roboflow/sports once cricket support is merged.
REPO_URL = "https://github.com/tanzzy45/sports.git"
REPO_BRANCH = "feat/cricket-pitch"
!git clone -q -b {REPO_BRANCH} {REPO_URL} {HOME}/sports

## Imports

In [ ]:
from google.colab import userdata
from IPython.display import Image

## Build dataset

The script downloads the three source datasets with your API key, renames their
keypoints to one schema, removes the augmented duplicates and splits the images by
video, so that near-identical neighbouring frames never end up in different splits.

In [ ]:
ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
DATASET = f"{HOME}/datasets/cricket-pitch"

!cd {HOME}/sports && ROBOFLOW_API_KEY={ROBOFLOW_API_KEY} PYTHONPATH=. python examples/cricket/prepare_pitch_dataset.py     --target_dir {DATASET} --download_dir {HOME}/datasets/downloads

## Custom training

- `mosaic=0.0` keeps the pitch geometry intact. Mosaic cuts the image into tiles, which
  breaks the perspective that the keypoints depend on.
- `imgsz=1280` because at the far end of the pitch the points are only a few pixels apart.
- Horizontal flips are on by default. `flip_idx` in `data.yaml` swaps the left and right
  keypoints when an image is flipped.
- For a better model, try `yolov8x-pose.pt` and a smaller batch.

In [ ]:
%cd {HOME}

!yolo task=pose mode=train model=yolov8m-pose.pt data={DATASET}/data.yaml batch=8 epochs=150 patience=30 imgsz=1280 mosaic=0.0 plots=True

In [ ]:
!ls {HOME}/runs/pose/train/

In [ ]:
%cd {HOME}
Image(filename=f'{HOME}/runs/pose/train/results.png', width=600)

In [ ]:
%cd {HOME}
Image(filename=f'{HOME}/runs/pose/train/val_batch0_pred.jpg', width=600)

## Validate custom model on the test split

In [ ]:
%cd {HOME}

!yolo task=pose mode=val model={HOME}/runs/pose/train/weights/best.pt data={DATASET}/data.yaml split=test imgsz=1280

## Measure the keypoint error

The metrics above score keypoints by similarity. What matters for the pitch map is how
far each point lands from its label compared with the size of the pitch, since that error
carries into the homography. The cell below reports it per keypoint on the test split.

In [ ]:
import glob
import json
import numpy as np
from ultralytics import YOLO

model = YOLO(f"{HOME}/runs/pose/train/weights/best.pt")
names = json.load(open(f"{DATASET}/schema.json"))["keypoints"]
errors = {name: [] for name in names}
labelled = {name: 0 for name in names}

for path in sorted(glob.glob(f"{DATASET}/images/test/*.jpg")):
    label = np.loadtxt(path.replace("images", "labels").replace(".jpg", ".txt"), ndmin=1)
    truth = label[5:].reshape(-1, 3)
    result = model(path, imgsz=1280, verbose=False)[0]
    height, width = result.orig_shape
    visible = truth[:, 2] > 0
    pitch_height = np.ptp(truth[visible, 1]) * height
    for i, name in enumerate(names):
        if not visible[i]:
            continue
        labelled[name] += 1
        if len(result.boxes) == 0:
            continue
        best = int(result.boxes.conf.argmax())
        prediction = result.keypoints.xyn[best][i].cpu().numpy()
        confidence = float(result.keypoints.conf[best][i]) if result.keypoints.conf is not None else 1.0
        if confidence < 0.5:
            continue
        error = np.linalg.norm((prediction - truth[i, :2]) * [width, height])
        errors[name].append(error / pitch_height)

print(f"{'keypoint':20} {'labelled':>8} {'found':>6} {'median error':>13}")
for name in names:
    found = errors[name]
    median = f"{np.median(found) * 100:.2f}%" if found else "-"
    print(f"{name:20} {labelled[name]:8} {len(found):6} {median:>13}")

## Save the model

Download `best.pt`, or copy it to Google Drive. `examples/cricket/pitch_map.py` will
use it to calibrate automatically.

In [ ]:
from google.colab import files

files.download(f"{HOME}/runs/pose/train/weights/best.pt")